In [31]:
import numpy as np
import pandas as pd
import moviepy 
import json
import os
import srt
import datetime
import subprocess

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip
from moviepy.video.io.VideoFileClip import VideoFileClip

def str_to_secs(time: str) -> float:
    'Amount of seconds for string in xx:xx:xx format'
    hours, mins, secs = time.replace(',','.').split(':')
    return float(hours)*3600 + float(mins)*60 + float(secs)

def secs_to_str(decimal_seconds):
    hours = decimal_seconds // 3600
    minutes = (decimal_seconds % 3600) // 60
    seconds = decimal_seconds % 60
    milliseconds = round((decimal_seconds - int(decimal_seconds)) * 1000)
    return "{:02d}:{:02d}:{:02d}.{:03d}".format(int(hours), int(minutes), int(seconds), int(milliseconds))

In [ ]:
class iLSUT_dataset(Dataset):
    
    # load data from csv file
    def __init__(self):
        self.df = pd.read_csv("../1_data/iLSU-T_video_IDs.csv") #print(self.df)
        self.video_IDs = self.df['video_ID']
        self.number_of_episodes = len(self.df['video_ID'])
        self.episode_paths = self.df['episode_path']
        self.fps = self.df['fps']
        self.duration = self.df['duration in frames']
        self.height = self.df['height']
        self.width = self.df['width']
        self.signer = self.df['signer']
        self.text = self.df['text']        
        return None

    # count the number of episodes or entries in the csv file
    def __len__(self):
        return self.number_of_episodes
    
    # return the i-th sample and the corresponding text transcription, given an index i
    def __getitem__(self, idx):
        print(self.video_IDs[idx])
        print(self.episode_paths[idx])
        print(self.fps[idx])
        print(self.duration[idx])
        print(self.height[idx])
        print(self.width[idx])
        print(self.signer[idx])
        print(self.text[idx])

        # get iLSU-T episode
        episode = VideoFileClip(self.episode_paths[idx])
        
        # get whisperx text transcription for the episode
        with open(self.text[idx],'r') as f:
            subs = json.load(f)

        # get the signer in the episode
        signer = self.signer[idx]
               
        return episode, subs, signer

    def generate_videos_with_one_signer(self, idx, version=1):
        
        if version == 1:
            prev_delta, post_delta = 0, 0 # time in seconds        

        episode_name = self.episode_paths[idx]
        episode = VideoFileClip(episode_name)
   
        with open(self.signer[idx],'r') as f:
            signer = json.load(f)
        
        periods = list(signer.keys())
        signer_begs, signer_ends, signer_list, N_periods = [], [], [], 0
        
        for period in periods:
            beg, end = str_to_secs(period.split('--')[0]), str_to_secs(period.split('--')[1])
            signer_begs.append(beg)
            signer_ends.append(end)            
            signer_list.append(signer[period])
            N_periods += 1

        source = episode_name[:-4].split('/')[-1].split('_')[0]
        video = episode_name[:-4].split('/')[-1].split('_')[1]
           
        for n in range(N_periods):
            video_clip_beg = signer_begs[n] - prev_delta
            if video_clip_beg < 0:
                video_clip_beg = 0
            
            video_clip_end = signer_ends[n] + post_delta
            if video_clip_end > self.duration[idx]/self.fps[idx]:
                video_clip_end = self.duration[idx]/self.fps[idx]
        
            video_clip = episode.subclip(video_clip_beg, video_clip_end)     
            video_clip_beg_str, video_clip_end_str = secs_to_str(video_clip_beg), secs_to_str(video_clip_end) 
                
            target_file = './videos_1/' + source + '_' + video + '_signer' + str(signer_list[n]) + '_' + video_clip_beg_str + '_' + video_clip_end_str + '.mp4'
            
            video_clip.write_videofile(target_file, audio=True, codec="libx264", fps=self.fps[idx])           
          
        return None

    
    def generate_video_clips(self, idx, version=1):
        
        if version == 1:            
            prev_delta, post_delta = 0.8 + 0.4*np.random.rand(), 1.8 + 0.4*np.random.rand()  # time in seconds        

        episode_name = self.episode_paths[idx]
        
   
        with open(self.text[idx],'r') as f:
            subs = json.load(f)
        
        source = episode_name[:-4].split('/')[-1].split('_')[0]
        print(source)
        video = episode_name[:-4].split('/')[-1].split('_')[1]
        print(video)

        # compose the video-clips for each episode
        for segment in subs['segments']:
            video_clip_beg = segment['start'] + prev_delta
            if video_clip_beg < 0:
                video_clip_beg = 0
            
            video_clip_end = segment['end'] + post_delta
            if video_clip_end > self.duration[idx]/self.fps[idx]:
                video_clip_end = self.duration[idx]/self.fps[idx]            
                  
            video_clip_beg_str, video_clip_end_str = secs_to_str(video_clip_beg), secs_to_str(video_clip_end) 

            try:
                os.makedirs("./video_clips/")
            except FileExistsError: 
                pass # directory already exists
                
            target_file = './video_clips/' + source + '_' + episode_name[16:] + '_' + video_clip_beg_str + '_' + video_clip_end_str + '.avi'

            comando = 'ffmpeg -i ' + episode_name + ' -c:v copy -ss ' + video_clip_beg_str + ' -to ' + video_clip_end_str + ' ' + target_file
            comando += ' -nostats'          
            
            subprocess.call(comando, shell=True)
                    
            video_clip_subs = []
            video_clip_subs.append(srt.Subtitle(index=0, start=datetime.timedelta(seconds=0), end=datetime.timedelta(seconds=video_clip_end - video_clip_beg),\
                                                     content=segment['text'], proprietary=''))
                    
            with open(target_file[:-4] + '.srt','w') as f:
                f.writelines(srt.compose(video_clip_subs))           
            
        return None

    def get_time_distribution_per_signer(self): # get the time distribution per signer in hours of episodes.
        
        signers = list(np.unique(np.array(self.signer)))
        time_per_signer = {}
        
        for signer_id in signers:
            time_acum = 0 # define a time accumulator
            
            for idx in range(self.number_of_episodes):
                if self.signer[idx] == signer_id:                          
                    time_acum += 1/3600*(self.duration[idx]/self.fps[idx])

            time_per_signer[signer_id] = time_acum 
            
        return time_per_signer
                

In [62]:
for n in range(iLSUT.number_of_episodes):
    iLSUT.generate_video_clips(idx=n, version=1)

1
PERIODISTAS


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

KeyboardInterrupt: 